# Simple RAG



In [103]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

import os, tiktoken
import pandas as pd

from tqdm import tqdm
import numpy as np

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import gradio as gr

load_dotenv(override=True)

URL_Reactome_Pathways = "https://download.reactome.org/95/ReactomePathways.txt"
URL_Reactome_Summary  = "https://reactome.org/download/current/pathway2summation.txt"

MODEL = "gpt-4.1-nano"

DB_NAME = "reactome_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

root_data = '../data'

openai = OpenAI()

### Get Reactome

In [68]:
def get_reactome_pathways(which:str="pathways", force:bool=False) -> pd.DataFrame:

    if which == "pathways":
        url = URL_Reactome_Pathways
        fname = "reactome_pathways.tsv"
        names=["reactome_id", "pathway", "species"]
    else:
        url = URL_Reactome_Summary
        fname = "reactome_summary.tsv"
        names=["reactome_id", "name", "summary", 'any2']

    filename = os.path.join(root_data, fname)

    if os.path.exists(filename) and not force:
        df = pd.read_csv(filename, sep="\t")
        print(f"File {filename} already exists: {df.shape}.")
        return df

    try:
        df = pd.read_csv(url,  sep="\t", header=None, names=names)
        df = df.drop(columns=['any2'], errors='ignore')
        df = df.iloc[1:]

    except Exception as e:
        print(f"Error fetching data from Reactome: {e}")
        return pd.DataFrame()
    
    df.to_csv(filename, sep="\t", index=False)

    return df

In [69]:
df = get_reactome_pathways(which="pathways", force=False)
print(len(df))

df.head(3)

File ../data/reactome_pathways.tsv already exists: (23289, 3).
23289


,reactome_id,pathway,species
0,R-BTA-73843,5-Phosphoribose 1-diphosphate biosynthesis,Bos taurus
1,R-BTA-1369062,ABC transporters in lipid homeostasis,Bos taurus
2,R-BTA-382556,ABC-family proteins mediated transport,Bos taurus


In [70]:
dfs = get_reactome_pathways(which="summary", force=False)
print(len(dfs))

df.head(3)

File ../data/reactome_summary.tsv already exists: (2849, 3).
2849


,reactome_id,pathway,species
0,R-BTA-73843,5-Phosphoribose 1-diphosphate biosynthesis,Bos taurus
1,R-BTA-1369062,ABC transporters in lipid homeostasis,Bos taurus
2,R-BTA-382556,ABC-family proteins mediated transport,Bos taurus


In [71]:
dfa = df[df.species == "Homo sapiens"]
print(len(dfa))

2848


In [72]:
dfs2 = dfs[dfs.reactome_id.isin(dfa.reactome_id)]
dfs2 = dfs2.drop_duplicates('reactome_id')
print(len(dfs2))

dfs2.tail(3)

2848


,reactome_id,name,summary
2846,R-HSA-199992,trans-Golgi Network Vesicle Budding,"After passing through the Golgi complex, secre..."
2847,R-HSA-192814,vRNA Synthesis,The synthesis of full-length negative strand v...
2848,R-HSA-192905,vRNP Assembly,"For each of eight gene segments, a viral ribon..."


In [73]:
text = ''

for i, row in dfs2.iterrows():
    text += f"{row['reactome_id']}  | {row['name']} + | + {row['summary']}\n"

print('----------end----------\n', i)

----------end----------
 2848


### Tokens

In [74]:
encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(text)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 909,137


### Documents

In [84]:
fname = "reactome_summary.tsv"
filename = os.path.join(root_data, fname)

if os.path.exists(filename):
    loader = TextLoader(filename, encoding="utf-8")
    documents = loader.load()
else:
    print("Nothing found")
    documents = []

print(documents[0])

page_content='reactome_id	name	summary
R-HSA-164843	2-LTR circle formation	The formation of 2-LTR circles requires the action of the cellular non-homologous DNA end-joining pathway.  Specifically the cellular Ku, XRCC4 and ligase IV proteins are needed.  Evidence for this is provided by the observation that cells mutant in these functions do not support detectable formation of 2-LTR circles, though integration and formation of 1-LTR circles are mostly normal.  The reaction takes place in the nucleus, and formation of 2-LTR circles has been used as a surrogate assay for nuclear transport.  It has also been suggested that the NHEJ system affects the toxicity of retroviral infection.
R-HSA-9909438	3-Methylcrotonyl-CoA carboxylase deficiency	3-methylcrotonyl-CoA carboxylase catalyzes the reversible conversion of 3-methylcrotonyl-CoA to 3-methylglutaconyl-CoA, the fourth step in the catabolism of leucine (Chu et al, 2007; Son et al, 2020). MCCC is composed of two subunits encoded by MCCC1 a

### Divide into chunks using the RecursiveCharacterTextSplitter

In [85]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=25)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 16590 chunks
First chunk:

page_content='reactome_id	name	summary' metadata={'source': '../data/reactome_summary.tsv'}


### Pick an embedding model

In [87]:
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(DB_NAME):
    # if the database already exists, delete it and create a new one
    # Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()
    print(f"Database {DB_NAME} already exists. Skipping creation.")
    vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
else:
    # langchain's Chroma wrapper for chromadb
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=DB_NAME)

print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 16590 documents


### Let's investigate the vectors

In [88]:

collection = vectorstore._collection
count = collection.count()

lista = collection.get(limit=1, include=["embeddings"])['embeddings']
print(type(lista))
sample_embedding = np.array(lista)[0]

dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

<class 'numpy.ndarray'>
There are 16,590 vectors with 3,072 dimensions in the vector store


#### Retriever

In [95]:
retriever = vectorstore.as_retriever()

### Vector similarity

In [91]:
question = "Innate Immune System"

retriever.invoke(question)

[Document(id='632c1cda-352d-40bd-90e2-31de028290a9', metadata={'source': '../data/reactome_summary.tsv'}, page_content="R-HSA-168249\tInnate Immune System\tInnate immunity encompases the nonspecific part of immunity tha are part of an individual's natural biologic makeup"),
 Document(id='ae33c6eb-814d-44e0-9eaf-ca6a883df4ac', metadata={'source': '../data/reactome_summary.tsv'}, page_content='hours and days of exposure to a new pathogen, our innate immune system.'),
 Document(id='5848bb00-fbf1-4827-83fe-ce9c3cde70e1', metadata={'source': '../data/reactome_summary.tsv'}, page_content='R-HSA-168643\tNucleotide-binding domain, leucine rich repeat containing receptor (NLR) signaling pathways\tThe innate immune system is the first line of defense against invading microorganisms, a broad specificity response characterized by the'),
 Document(id='365f4022-419d-4992-b778-ce7d943d7891', metadata={'source': '../data/reactome_summary.tsv'}, page_content='(PMN) leukocytes (i.e., neutrophils) and mo

### RAG

In [99]:
llm = ChatOpenAI(temperature=0, model=MODEL)

SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, a Reactome assistant.
You are chatting with a user about Reactome's pathways.
If relevant, use the given context to answer any scientific question.
If you don't know the answer, say so.
Context:
{context}
"""

In [116]:
def answer_question(question: str, history: list) -> str:
    # Retrieve relevant documents from the vector store
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    # Format the system prompt with the retrieved context
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)

    # Invoke the LLM with the system prompt and the user's question
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return str(response.content)



In [ ]:
question = "Does a bacteria invasion elicits an innate immune response? Which are the main pathways involved?"
answer_question(question, [])

'Yes, bacterial invasion typically elicits an innate immune response. The innate immune system recognizes bacterial components through pattern recognition receptors (PRRs), such as Toll-like receptors (TLRs) and Nucleotide-binding domain, leucine rich repeat containing receptors (NLRs). \n\nThe main pathways involved include:\n- **NLR signaling pathways (R-HSA-168643)**, which detect intracellular bacterial components and activate inflammatory responses.\n- **Toll-like receptor pathways**, which recognize microbial PAMPs on the cell surface or within endosomes, leading to the activation of downstream signaling cascades that promote cytokine production and immune cell recruitment.\n\nThese pathways coordinate the recruitment and activation of phagocytes and the release of antimicrobial peptides to contain and eliminate the bacteria.'

In [118]:
gr_question = gr.ChatInterface(
    fn=answer_question,
    chatbot=gr.Chatbot(type="messages"),
)

gr_question.launch()

/home/flavio/uv/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:330: UserWarning: The gr.ChatInterface was not provided with a type, so the type of the gr.Chatbot, 'messages', will be used.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
